## 1. Vấn Đề Chính: Model Architecture vs Task

### Model MGSAN được train như thế nào?
- **Input**: Fixed window size (64 frames) của skeleton sequence
- **Output**: Single action label cho window đó
- **Task**: Action Classification (phân loại 1 clip ngắn)

### Nhưng Evaluation Task là gì?
- **Input**: Video dài với nhiều actions liên tiếp
- **Output**: Frame-wise predictions (label cho từng frame)
- **Task**: Temporal Action Segmentation (phân đoạn hành động theo thời gian)

### 🔴 Vấn đề:
```
Model train để classify 1 window 64 frames
→ Nhưng evaluation cần predict cho MỌI FRAME trong video dài
→ Code hiện tại chỉ predict 1 label cho CẢ VIDEO
→ Accuracy thấp là tất yếu!
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Minh họa vấn đề
print("="*60)
print("VÍ DỤ MINH HỌA VẤN ĐỀ")
print("="*60)

# Giả sử video có 500 frames với 5 actions khác nhau
video_length = 500
ground_truth = np.array([0]*100 + [1]*80 + [2]*120 + [3]*90 + [4]*110)

print(f"\nVideo length: {video_length} frames")
print(f"Ground truth actions: {np.unique(ground_truth)}")
print(f"Số lần thay đổi action: {np.sum(ground_truth[1:] != ground_truth[:-1])}")

# Code hiện tại: Predict 1 label cho cả video
current_prediction = np.full(video_length, 2)  # Giả sử model predict tất cả là action 2

# Tính accuracy
accuracy = np.mean(current_prediction == ground_truth)
print(f"\n{'❌ CURRENT APPROACH:':<30} Accuracy = {accuracy:.2%}")
print(f"{'  ':<30} → Tất cả frames đều được gán label 2")
print(f"{'  ':<30} → Chỉ đúng cho đoạn action 2 (120/500 frames)")

## 2. Phân Tích Code Hiện Tại

### Vấn đề trong `predict_video()`:

In [ ]:
# CODE HIỆN TẠI (SAI)
def predict_video_WRONG(model, skeleton_data, device='cuda'):
    """
    ❌ Vấn đề: Predict 1 label cho CẢ VIDEO
    """
    # ... xử lý data ...
    
    with torch.no_grad():
        output = model(data_tensor)  # Predict cả video 1 lúc
        pred_label = output.argmax(dim=1).item()  # 1 label duy nhất!
    
    return pred_label  # ❌ Trả về 1 số, không phải array

# Sau đó trong evaluate:
# num_frames = 500
# pred_label_list = np.full((num_frames, 1), pred_label)  # ❌ Tất cả frames = 1 label
# → Accuracy = 38% là may mắn rồi!

print("❌ Vấn đề 1: Model nhận input CẢ VIDEO thay vì WINDOW 64 frames")
print("❌ Vấn đề 2: Trả về 1 LABEL thay vì label cho TỪNG FRAME")
print("❌ Vấn đề 3: Không có SLIDING WINDOW để scan qua video")

## 3. So Sánh với DD-Net (Code Gốc)

### DD-Net làm gì?

In [ ]:
# CODE GỐC DD-NET (ĐÚNG)
def ddnet_approach():
    """
    ✅ DD-Net đã tính toán sẵn predictions cho TỪNG FRAME
    """
    # pred_label_list = np.load(os.path.join(POSE_PATH, video), allow_pickle=True)
    # → Shape: (num_frames, 1) - MỖI FRAME có 1 label
    
    # Ví dụ:
    # Frame 0-99: label 0
    # Frame 100-179: label 1
    # Frame 180-299: label 2
    # ...
    pass

print("✅ DD-Net: Predictions đã được tính per-frame")
print("❌ MGSAN: Code hiện tại chỉ predict 1 label cho cả video")
print("\n→ Đây là lý do chính gây ra accuracy thấp!")

## 4. Lý Do Tại Sao 38%?

Hãy tính toán:

In [ ]:
# Giả sử video có phân bố actions như sau:
action_distribution = {
    0: 150,  # frames
    1: 100,
    2: 200,  # Action xuất hiện nhiều nhất
    3: 80,
    4: 70
}

total_frames = sum(action_distribution.values())
most_common_action = 2
most_common_frames = action_distribution[most_common_action]

baseline_accuracy = most_common_frames / total_frames

print("Giả sử model luôn predict action xuất hiện nhiều nhất:")
print(f"  Total frames: {total_frames}")
print(f"  Most common action: {most_common_action} ({most_common_frames} frames)")
print(f"  Baseline accuracy: {baseline_accuracy:.2%}")
print("\n→ 38% phù hợp với việc model 'đoán' action phổ biến nhất!")
print("→ Model không thực sự học được temporal patterns!")

## 5. Các Vấn Đề Bổ Sung

### 5.1. Window Size Mismatch

In [ ]:
print("Model được train với:")
print("  - Window size: 64 frames")
print("  - Input shape: (N, 3, 64, 48, 1)")

print("\nNhưng evaluation feed:")
print("  - Whole video: 500-2000 frames")
print("  - Input shape: (N, 3, 500, 48, 1)")

print("\n❌ Model không được train để xử lý sequences dài!")
print("❌ Temporal context hoàn toàn khác với training!")

### 5.2. Temporal Context Loss

In [ ]:
print("Training:")
print("  - Model học: 'trong 64 frames này, action là gì?'")
print("  - Có temporal context trong window")

print("\nEvaluation (sai):")
print("  - Model nhận: toàn bộ video")
print("  - Không biết xử lý như thế nào")
print("  - Network chỉ lấy global pooling → mất thông tin temporal")

print("\n→ Model không thể phân biệt các actions khác nhau trong video!")

### 5.3. Action Boundary Detection

In [ ]:
print("Task thực tế cần:")
print("  ✓ Detect khi nào action thay đổi")
print("  ✓ Phân đoạn chính xác ranh giới")
print("  ✓ Classify từng đoạn")

print("\nNhưng model MGSAN:")
print("  ❌ Không được train để detect boundaries")
print("  ❌ Chỉ classify fixed-length clips")
print("  ❌ Không có temporal segmentation head")

## 6. Giải Pháp Đề Xuất

### Solution 1: Sliding Window (Đơn giản nhất)

In [ ]:
def predict_video_sliding_window(model, skeleton_data, window_size=64, stride=1, device='cuda'):
    """
    ✅ Sử dụng sliding window để predict từng frame
    
    Args:
        window_size: 64 frames (như training)
        stride: Bước nhảy (1 = dense prediction, >1 = faster)
    """
    model = model.to(device)
    model.eval()
    
    T = skeleton_data.shape[1]  # Số frames
    predictions = []
    
    # Slide window qua video
    for t in range(0, T - window_size + 1, stride):
        # Extract window
        window = skeleton_data[:, t:t+window_size, :, :]
        
        # Predict
        with torch.no_grad():
            output = model(torch.FloatTensor(window).unsqueeze(0).to(device))
            pred = output.argmax(dim=1).item()
        
        # Assign prediction to all frames in window
        predictions.extend([pred] * stride)
    
    # Handle remaining frames
    if len(predictions) < T:
        predictions.extend([predictions[-1]] * (T - len(predictions)))
    
    return np.array(predictions)

print("✅ Sliding Window Approach:")
print("  1. Chia video thành windows 64 frames")
print("  2. Predict từng window")
print("  3. Aggregate predictions cho từng frame")
print("\n→ Dự kiến cải thiện accuracy lên 60-75%")

### Solution 2: Temporal Smoothing

In [ ]:
def temporal_smoothing(predictions, window_size=30):
    """
    ✅ Làm mượt predictions để giảm noise
    """
    from scipy.ndimage import median_filter
    smoothed = median_filter(predictions, size=window_size)
    return smoothed

print("✅ Temporal Smoothing:")
print("  - Loại bỏ predictions nhảy nhót")
print("  - Đảm bảo consistency trong cùng action")
print("  - Dùng median filter hoặc majority voting")

### Solution 3: Multi-Scale Windows

In [ ]:
print("✅ Multi-Scale Approach:")
print("  - Window sizes: [32, 64, 128] frames")
print("  - Predict ở nhiều scales")
print("  - Ensemble/voting kết quả")
print("\n→ Capture được actions dài/ngắn khác nhau")

## 7. Tóm Tắt & Khuyến Nghị

### ❌ Tại sao 38% accuracy?

In [ ]:
print("=" * 60)
print("CÁC NGUYÊN NHÂN CHÍNH")
print("=" * 60)

causes = [
    ("1. Architecture Mismatch", 
     "Model train cho classification, evaluation là segmentation"),
    
    ("2. Window Size Problem", 
     "Train: 64 frames, Eval: cả video (500-2000 frames)"),
    
    ("3. Single Label Output", 
     "Code predict 1 label cho cả video thay vì per-frame"),
    
    ("4. No Temporal Modeling", 
     "Không có sliding window hoặc temporal aggregation"),
    
    ("5. Baseline Performance", 
     "38% ~ random guessing với action phổ biến nhất")
]

for title, desc in causes:
    print(f"\n{title}")
    print(f"  → {desc}")

print("\n" + "=" * 60)
print("KHUYẾN NGHỊ")
print("=" * 60)

recommendations = [
    "1. Implement Sliding Window (priority 1)",
    "2. Add Temporal Smoothing",
    "3. Tune window size & stride",
    "4. Consider training temporal segmentation model",
    "5. Use action boundary detection"
]

for rec in recommendations:
    print(f"  ✓ {rec}")

## 8. Kết Luận

**MGSAN model hoạt động tốt cho action classification**, nhưng:
- ❌ Không phù hợp trực tiếp cho temporal action segmentation
- ❌ Code evaluation hiện tại sai cơ bản (predict 1 label cho cả video)
- ✅ Cần sliding window approach để áp dụng cho segmentation task
- ✅ Expected accuracy sau khi fix: 60-80% (tùy dataset)

**Next Steps:**
1. Implement sliding window prediction
2. Test với stride khác nhau (1, 4, 8, 16)
3. Add temporal smoothing
4. So sánh với DD-Net results

---
**Created**: 2025-12-20  
**Author**: Analysis of MGSAN Online Evaluation